In [1]:
import torch
print(torch.cuda.is_available())

True


In [ ]:
import torch
import numpy as np

tensor는 array, matrix왜 매우 유사한 특수한 자료구조

pytorch에서는 텐서를 사용하여 모델의 input, output, 매개변수들을 encoding

tensor는 gpu나 tpu에서도 사용할 수 있다는 점만 제외하면 NumPy의 ndarray와 유사

tensor 초기화 

In [3]:
# data로부터 직접 생성하기 (data type은 자동으로 유추)
data = [[1,2],[3,4]]
x_data = torch.tensor(data)
print(f"x_data:\n {x_data}\n")


x_data:
 tensor([[1, 2],
        [3, 4]])



In [4]:
# NumPy 배열로부터 생성하기 (반대도 가능)
np_array = np.array(data)
x_np = torch.from_numpy(np_array)
print(f"x_np:\n {x_np}\n")

x_np:
 tensor([[1, 2],
        [3, 4]])



In [5]:
# 다른 tensor로부터 생성하기 (명시적 재정의 없으면 기존 텐서의 shape, datatype 유지)

#----------------------------------------------------#
# 모든 원소가 1인 tensor 만들기
# torch.ones_like(input, *, dtype=None, layout=None, device=None,
#                  requires_grad=False, memory_format=torch.preserve_format)
# 필수 인자 input - 참조할 tensor (shape을 따라감)
# 선택 키워드 인자 dtype — 결과 텐서의 자료형, 지정하지 않으면 input과 동일
#                layout — 메모리 레이아웃 (torch.strided 등)
#                device — 텐서가 올라갈 장치 ('cpu', 'cuda' 등)
#                requires_grad — autograd 추적 여부 (기본 False)
#                memory_format — 메모리 포맷 (기본 torch.preserve_format)
x_ones = torch.ones_like(x_data)
print(f"Ones Tensors:\n {x_ones}\n")
#----------------------------------------------------#
# 원소가 0~1 사이인 tensor 만들기
# torch.rand_like(input, *, dtype=None, layout=None, device=None,
#                  requires_grad=False, memory_format=torch.preserve_format)
# 0~1 사이 실수 난수 생성이기 때문에 int 타입일 시에 오류 발생
x_rand = torch.rand_like(x_data, dtype=torch.float)
print(f"Random Tensor:\n {x_rand}\n")

Ones Tensors:
 tensor([[1, 1],
        [1, 1]])



Random Tensor:
 tensor([[0.9712, 0.4014],
        [0.3333, 0.7348]])



In [6]:
# random 또는 constant 값을 사용하기

# shape: tensor의 dimension을 나타내는 tuple
shape = (2,3,)
# 끝의 comma 하나는 trailing comma라고 불림
# 일반적으로 기능에는 문제가 없고 편의를 위해 사용하지만
# 1차원 tensor의 경우에는 차이가 존재함
#       ex) a = (5)     그냥 정수 5
#       ex) a = (5,)    원소가 한개인 tuple
rand_tensor = torch.rand(shape)
ones_tensor = torch.ones(shape)
zeros_tensor = torch.zeros(shape)

print(f"Random Tensor:\n {rand_tensor}\n")
print(f"Ones Tensor:\n {ones_tensor}\n")
print(f"Zeros Tensor:\n {zeros_tensor}\n")

Random Tensor:
 tensor([[0.6326, 0.7520, 0.2280],
        [0.0878, 0.7360, 0.1456]])

Ones Tensor:
 tensor([[1., 1., 1.],
        [1., 1., 1.]])

Zeros Tensor:
 tensor([[0., 0., 0.],
        [0., 0., 0.]])



텐서의 속성(Attribute)

In [7]:
tensor = torch.rand(3,4)

print(f"Shape of tensor: {tensor.shape}")
print(f"Datatype of tensor: {tensor.dtype}")
print(f"Device tensor is stored on: {tensor.device}")

Shape of tensor: torch.Size([3, 4])
Datatype of tensor: torch.float32
Device tensor is stored on: cpu


텐서 연산 (Operation)

In [8]:
# GPU가 존재하면 텐서를 이동한다
# C의 삼항연산자와 동일한 문법
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [24]:
# NumPy식의 표준 인덱싱과 슬라이싱

tensor = torch.ones(4, 4)

print(f"First row: {tensor[0]}")        # tensor의 0행 출력
print(f"First column: {tensor[:, 0]}")  # tensor의 0열 출력
# ... -> 앞 차원을 모두 :로 반영
# -1  -> 파이썬의 음수 인덱스는 뒤에서 부터 셈 (-1: 마지막, -2: 마지막에서 하나 전, ...)
print(f"Last column: {tensor[..., -1]}")# tensor의 마지막 열 출력

tensor[:, 1] = 0 # 두번째 열을 모두 0으로
print(tensor)

First row: tensor([1., 1., 1., 1.])
First column: tensor([1., 1., 1., 1.])
Last column: tensor([1., 1., 1., 1.])
tensor([[1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.]])


In [10]:
# tensor 합치기
t1 = torch.cat([tensor,tensor,tensor], dim=1)
print(t1)
print(f"Size of tensor 't1': {t1.shape}")

tensor([[1., 0., 1., 1., 1., 0., 1., 1., 1., 0., 1., 1.],
        [1., 0., 1., 1., 1., 0., 1., 1., 1., 0., 1., 1.],
        [1., 0., 1., 1., 1., 0., 1., 1., 1., 0., 1., 1.],
        [1., 0., 1., 1., 1., 0., 1., 1., 1., 0., 1., 1.]])
Size of tensor 't1': torch.Size([4, 12])


In [11]:
# Arithmetic operations

# 두 텐서 간의 MM을 계산한다. 
# y1, y2, y3는 모두 같은 연산을 통해 같은 값을 갖는다
y1 = tensor @ tensor.T  # tensor.T는 전치행렬(Transpose)
                        # @는 Matrix Multiplication 연산자
print(f"y1:\n{y1}\n")
y2 = tensor.matmul(tensor.T)
print(f"y2:\n{y2}\n")
y3 = torch.rand_like(y1)
torch.matmul(tensor, tensor.T, out=y3)
print(f"y3:\n{y3}\n")


y1:
tensor([[3., 3., 3., 3.],
        [3., 3., 3., 3.],
        [3., 3., 3., 3.],
        [3., 3., 3., 3.]])

y2:
tensor([[3., 3., 3., 3.],
        [3., 3., 3., 3.],
        [3., 3., 3., 3.],
        [3., 3., 3., 3.]])

y3:
tensor([[3., 3., 3., 3.],
        [3., 3., 3., 3.],
        [3., 3., 3., 3.],
        [3., 3., 3., 3.]])



In [15]:
# single-element tensor

agg = tensor.sum()      # 원소의 합 계산하기 torch로 반환됨
agg_item = agg.item()   # .item()은 원소 하나짜리인 tensor를 python 숫자 값으로 변환 
print(agg_item, type(agg_item))

12.0 <class 'float'>


In [ ]:
# in-place operations
# '_' 접미사가 붙는 함수들은 연산 결과를 피연산자에 저장하는 바꿔치기 연산
# 바꿔치기 연산은 메모리를 절약할 수 있지만 history가 즉시 삭제되어 
# derivative 계산에 문제가 발생할 수 있으므로 사용 권장하지 않음

print(tensor, "\n")
tensor.add_(5)
print(tensor)

tensor([[1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.],
        [1., 0., 1., 1.]]) 

tensor([[6., 5., 6., 6.],
        [6., 5., 6., 6.],
        [6., 5., 6., 6.],
        [6., 5., 6., 6.]])


NumPy 변환 (Bridge)

In [32]:
# 텐서를 NumPy 배열로 변환하기

t = torch.ones(5)
print(f"t: {t}")
n = t.numpy()
print(f"n: {n}")

# tensor와 numpy 배열은 같은 메모리를 공유하므로 변경사항 반영
t.add_(1)
print(f"t: {t}")
print(f"n: {n}")

t: tensor([1., 1., 1., 1., 1.])
n: [1. 1. 1. 1. 1.]
t: tensor([2., 2., 2., 2., 2.])
n: [2. 2. 2. 2. 2.]


In [35]:
# NumPy 배열을 텐서로 변환하기

n = np.ones(5)
t = torch.from_numpy(n)

np.add(n, 1, out = n)
print(f"t: {t}")
print(f"n: {n}")

t: tensor([2., 2., 2., 2., 2.], dtype=torch.float64)
n: [2. 2. 2. 2. 2.]
